
# 練習問題: Boids (群れの創発)

## 目標

鳥や魚の「群れ」は, 1羽1羽が近くの仲間を見て簡単なルールに従うだけで, 全体としてまとまった動きが**自然に立ち上がる (創発する)**。各個体が近傍の全個体を見るので計算は O(N²)。各個体の更新は独立なので, 外側ループを `parallel for` で並列化する典型例。

## 課題

`boids.cpp` (または `boids.f90`) は, 各個体について近傍の仲間を調べ, 3つのルール

- **結合 (cohesion)**: 近傍の重心へ向かう
- **整列 (alignment)**: 近傍の平均速度に合わせる
- **分離 (separation)**: 近すぎる相手から離れる

で速度を更新し, 位置を進める (周期境界)。現在と次の状態を別の配列に持つので, 各個体の更新は互いに独立。

`TODO` の箇所に **OpenMP の指示を追加** し, 各個体 `i` のループを並列化せよ。

- C++: 外側 `for` の直前に `#pragma omp parallel for` を1行加える。
- Fortran: 外側 `do` を `!$omp parallel do private(...)` と `!$omp end parallel do` で囲む (一時変数を `private` に。ひな形に記入済み)。

ポイント:

- 並列化するのは**個体 `i` の外側ループ**。読みは現在の配列, 書きは次の配列なので競合しない。
- 更新は決定的なので, **スレッド数を変えても結果は完全に同じ**になる。

## コンパイルと実行

```
# C++
nvc++ -fast -mp=multicore boids.cpp -o boids.exe

# Fortran
nvfortran -fast -mp=multicore boids.f90 -o boids.exe
```

第1引数で個体数 `N`, 第2引数で時間ステップ数。

```
OMP_NUM_THREADS=4 ./boids.exe 1000 300
```

## 期待される結果

```
N=1000, steps=300: 整列度 0.0347 -> 0.9760 (1 に近いほど群れが揃った)
```

- **整列度 (polarization)** は, 全個体の進行方向の揃い具合 (0=バラバラ, 1=完全に同じ向き)。
- 初めはランダムな向きなので整列度はほぼ 0。シミュレーションが進むと, **誰も「全体で揃え」と命令していないのに**, 群れ全体が同じ方向を向くようになり整列度が 1 に近づく。これが**創発**。
- スレッド数を変えても整列度は同じになる (決定的な計算なので)。
- `N` を増やすと計算は O(N²) で重くなる。`parallel for` の台数効果を「性能を比べる」セルで測ってみよ。
- (発展) 整列の強さ `wa` や近傍半径 `R` を小さくすると, 群れが1つにまとまらず整列度が上がりにくくなる。パラメータを変えて挙動の違いを観察せよ。



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ boids.cpp
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <omp.h>

/* 状態を持たない乱数 (初期配置の再現性のため): (seed,k) から [0,1)。 */
static inline double draw_rand01(long long seed, long long k) {
  const long long M = 2147483647LL;
  long long x = ((seed % M) * 2654435761LL + (k % M) + 1) % M;
  x = ((x ^ (x >> 16)) * 1812433253LL) % M;
  x = ((x ^ (x >> 13)) * 1664525LL)    % M;
  x =  (x ^ (x >> 16)) % M;
  return (double)x / (double)M;
}

/* 群れの「整列度」(polarization): 全個体の進行方向の平均の大きさ。
   バラバラなら 0 に近く, みんな同じ向きなら 1 に近い。 */
double polarization(int N, const double * vx, const double * vy) {
  double sx = 0.0, sy = 0.0;
  for (int i = 0; i < N; i++) {
    double s = sqrt(vx[i]*vx[i] + vy[i]*vy[i]);
    sx += vx[i] / s; sy += vy[i] / s;
  }
  return sqrt(sx*sx + sy*sy) / N;
}

int main(int argc, char ** argv) {
  int N     = (argc > 1 ? atoi(argv[1]) : 1000);   /* 個体数 */
  int steps = (argc > 2 ? atoi(argv[2]) : 300);    /* 時間ステップ数 */
  /* ルールのパラメータ */
  double box = 30.0;        /* 正方形領域 (周期境界) */
  double R = 15.0, Rs = 2.0;/* 近傍半径, 分離半径 */
  double wc = 0.01, wa = 0.2, ws = 0.05;  /* 結合, 整列, 分離の強さ */
  double speed = 1.0, dt = 1.0;

  /* 現在(px..) と 次(qx..) の2組の配列 (Jacobi のように読みと書きを分ける) */
  double * px = (double *)malloc(sizeof(double) * N), * py = (double *)malloc(sizeof(double) * N);
  double * vx = (double *)malloc(sizeof(double) * N), * vy = (double *)malloc(sizeof(double) * N);
  double * qx = (double *)malloc(sizeof(double) * N), * qy = (double *)malloc(sizeof(double) * N);
  double * ux = (double *)malloc(sizeof(double) * N), * uy = (double *)malloc(sizeof(double) * N);
  for (int i = 0; i < N; i++) {
    px[i] = box * draw_rand01(i, 0);
    py[i] = box * draw_rand01(i, 1);
    double a = 2.0 * M_PI * draw_rand01(i, 2);   /* ランダムな初期方向 */
    vx[i] = cos(a); vy[i] = sin(a);
  }
  double P0 = polarization(N, vx, vy);

  double t0 = omp_get_wtime();
  for (int t = 0; t < steps; t++) {
    /* 各個体 i を, 近傍 j を見て更新する (近傍探索が O(N), 全体で O(N^2))。 */
    // TODO: 各個体 i のループを #pragma omp parallel for で並列化せよ (i ごとに独立)。
    for (int i = 0; i < N; i++) {
      double cx = 0, cy = 0, avx = 0, avy = 0, sx = 0, sy = 0;
      int cnt = 0;
      for (int j = 0; j < N; j++) {
        if (j == i) continue;
        double dx = px[j] - px[i], dy = py[j] - py[i];
        double d2 = dx*dx + dy*dy;
        if (d2 < R * R) {
          cx += px[j]; cy += py[j]; avx += vx[j]; avy += vy[j]; cnt++;
          if (d2 < Rs * Rs) { sx += px[i] - px[j]; sy += py[i] - py[j]; }  /* 分離: 近すぎる相手から離れる */
        }
      }
      double ax = 0, ay = 0;
      if (cnt > 0) {
        cx /= cnt; cy /= cnt; avx /= cnt; avy /= cnt;
        ax += wc * (cx - px[i]) + wa * (avx - vx[i]);   /* 結合 + 整列 */
        ay += wc * (cy - py[i]) + wa * (avy - vy[i]);
      }
      ax += ws * sx; ay += ws * sy;                     /* 分離 */
      double nvx = vx[i] + ax, nvy = vy[i] + ay;
      double s = sqrt(nvx*nvx + nvy*nvy); if (s < 1e-9) s = 1.0;
      nvx = nvx / s * speed; nvy = nvy / s * speed;     /* 速さは一定に保つ */
      ux[i] = nvx; uy[i] = nvy;
      qx[i] = fmod(px[i] + nvx * dt + box, box);        /* 周期境界 (はみ出たら反対側へ) */
      qy[i] = fmod(py[i] + nvy * dt + box, box);
    }
    /* 現在 <-> 次 を入れ替える */
    double * t1;
    t1 = px; px = qx; qx = t1;  t1 = py; py = qy; qy = t1;
    t1 = vx; vx = ux; ux = t1;  t1 = vy; vy = uy; uy = t1;
  }
  double elapsed = omp_get_wtime() - t0;

  printf("N=%d, steps=%d: 整列度 %.4f -> %.4f (1 に近いほど群れが揃った)\n",
         N, steps, P0, polarization(N, vx, vy));
  printf("elapsed = %.3f sec\n", elapsed);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore boids.cpp -o boids_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./boids_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ boids.f90
module boids_mod
contains
  ! 状態を持たない乱数 (初期配置の再現性のため): (seed,k) から [0,1)。
  function draw_rand01(seed, k) result(u)
    integer(8), intent(in) :: seed, k
    real(8) :: u
    integer(8), parameter :: M = 2147483647_8
    integer(8) :: x
    x = modulo(modulo(seed, M) * 2654435761_8 + modulo(k, M) + 1_8, M)
    x = modulo(ieor(x, ishft(x, -16)) * 1812433253_8, M)
    x = modulo(ieor(x, ishft(x, -13)) * 1664525_8,    M)
    x = modulo(ieor(x, ishft(x, -16)), M)
    u = real(x, 8) / real(M, 8)
  end function draw_rand01

  ! 群れの整列度 (polarization): 全個体の進行方向の平均の大きさ (0=バラバラ, 1=揃った)。
  function polarization(N, vx, vy) result(P)
    integer, intent(in) :: N
    real(8), intent(in) :: vx(N), vy(N)
    real(8) :: P, sx, sy, s
    integer :: i
    sx = 0.0d0; sy = 0.0d0
    do i = 1, N
       s = sqrt(vx(i)**2 + vy(i)**2)
       sx = sx + vx(i)/s; sy = sy + vy(i)/s
    end do
    P = sqrt(sx*sx + sy*sy) / N
  end function polarization
end module boids_mod

program boids
  use boids_mod
  use omp_lib
  character(len=32) :: arg
  integer :: N, steps, t, i, j, cnt
  real(8) :: box, R, Rs, wc, wa, ws, speed, dt, P0
  real(8) :: cx, cy, avx, avy, sx, sy, ax, ay, dx, dy, d2, nvx, nvy, s, ang, t0, elapsed
  real(8), allocatable :: px(:), py(:), vx(:), vy(:), qx(:), qy(:), ux(:), uy(:), tmp(:)
  N = 1000; steps = 300
  box = 30.0d0; R = 15.0d0; Rs = 2.0d0
  wc = 0.01d0; wa = 0.2d0; ws = 0.05d0; speed = 1.0d0; dt = 1.0d0
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) N
  end if
  if (command_argument_count() >= 2) then
     call get_command_argument(2, arg); read (arg, *) steps
  end if

  allocate(px(N), py(N), vx(N), vy(N), qx(N), qy(N), ux(N), uy(N))
  do i = 1, N
     px(i) = box * draw_rand01(int(i-1,8), 0_8)
     py(i) = box * draw_rand01(int(i-1,8), 1_8)
     ang = 2.0d0 * 3.14159265358979d0 * draw_rand01(int(i-1,8), 2_8)
     vx(i) = cos(ang); vy(i) = sin(ang)
  end do
  P0 = polarization(N, vx, vy)

  t0 = omp_get_wtime()
  do t = 1, steps
     ! 各個体 i を, 近傍 j を見て更新する (全体で O(N^2))。
     ! TODO: 各個体 i のループを !$omp parallel do (i ごとに独立) で並列化せよ。
     do i = 1, N
        cx = 0; cy = 0; avx = 0; avy = 0; sx = 0; sy = 0; cnt = 0
        do j = 1, N
           if (j == i) cycle
           dx = px(j) - px(i); dy = py(j) - py(i); d2 = dx*dx + dy*dy
           if (d2 < R*R) then
              cx = cx + px(j); cy = cy + py(j); avx = avx + vx(j); avy = avy + vy(j); cnt = cnt + 1
              if (d2 < Rs*Rs) then     ! 分離: 近すぎる相手から離れる
                 sx = sx + (px(i) - px(j)); sy = sy + (py(i) - py(j))
              end if
           end if
        end do
        ax = 0; ay = 0
        if (cnt > 0) then
           cx = cx/cnt; cy = cy/cnt; avx = avx/cnt; avy = avy/cnt
           ax = ax + wc*(cx - px(i)) + wa*(avx - vx(i))   ! 結合 + 整列
           ay = ay + wc*(cy - py(i)) + wa*(avy - vy(i))
        end if
        ax = ax + ws*sx; ay = ay + ws*sy                  ! 分離
        nvx = vx(i) + ax; nvy = vy(i) + ay
        s = sqrt(nvx*nvx + nvy*nvy); if (s < 1d-9) s = 1.0d0
        nvx = nvx/s*speed; nvy = nvy/s*speed               ! 速さは一定に
        ux(i) = nvx; uy(i) = nvy
        qx(i) = modulo(px(i) + nvx*dt, box)                ! 周期境界
        qy(i) = modulo(py(i) + nvy*dt, box)
     end do
     ! TODO: 上で始めた parallel do 領域を閉じる (!$omp end parallel do)。
     ! 現在 <-> 次 を入れ替える
     call move_alloc(px, tmp); call move_alloc(qx, px); call move_alloc(tmp, qx)
     call move_alloc(py, tmp); call move_alloc(qy, py); call move_alloc(tmp, qy)
     call move_alloc(vx, tmp); call move_alloc(ux, vx); call move_alloc(tmp, ux)
     call move_alloc(vy, tmp); call move_alloc(uy, vy); call move_alloc(tmp, uy)
  end do
  elapsed = omp_get_wtime() - t0

  print "(a,i0,a,i0,a,f0.4,a,f0.4,a)", &
       "N=", N, ", steps=", steps, ": 整列度 ", P0, " -> ", polarization(N, vx, vy), &
       " (1 に近いほど群れが揃った)"
  print "(a,f0.3,a)", "elapsed = ", elapsed, " sec"
end program boids

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore boids.f90 -o boids_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./boids_f90.exe


# 4. 発展目標 (できる範囲で挑戦)
- この問題の基本は **マルチコア並列化** (`#pragma omp parallel for` / `!$omp parallel do` など)。まずはここまでを目指す。
- 余力があれば次にも挑戦:
  - **GPUで並列化**: コンパイルを `-mp=gpu` にして, 重いループに `#pragma omp target teams distribute parallel for` (+ 必要に応じて `map`) を付ける。
  - **SIMD化** (11/12章): 内側ループに `#pragma omp simd`, またはベクトル型。 **ILP向上** (13章): ベクトル長 `nl` の調整。
- どこまで速くできるか挑戦し, その効果を下の「性能を比べる」で可視化しよう。

# 5. 性能を比べる (任意)
- 各プログラムは主計算部分の所要時間を `elapsed = ... sec` の形で表示する。
- 構成を変えて (スレッド数, SIMDの有無, GPU など) 実行し, 得られた **経過秒** を下の `DATA` に「ラベルと秒」で並べて実行すると, 棒グラフと「基準(先頭)比のスピードアップ」が出る。
- 試した構成だけ書けばよい (`#` で始まる行は無視)。


In [ ]:
import matplotlib.pyplot as plt

# 各構成の (ラベル, 経過秒) を並べる。先頭の行を基準(スピードアップ=1)にする。
# 自分が実際に試した構成の数値に書き換える。
DATA = [
    ("serial",    1.00),
    ("omp-8",     0.20),
    # ("simd",      0.50),
    # ("simd+omp",  0.07),
    # ("gpu",       0.05),
]

labels = [d[0] for d in DATA]
secs   = [float(d[1]) for d in DATA]
speed  = [secs[0] / s for s in secs]                 # 先頭(基準)比のスピードアップ
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].bar(labels, secs)
ax[0].set_ylabel("elapsed (s)")
ax[0].set_title("time (lower = faster)")
ax[1].bar(labels, speed)
ax[1].set_ylabel("speedup vs " + labels[0])
ax[1].set_title("speedup (higher = faster)")
for a in ax:
    a.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


# 6. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:boids.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 6-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 6-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 6-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:boids.cpp}

コマンドと実行結果:
{bash[-1]}

## 6-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:boids.cpp}